# 05 — The estimator & its parameters

*Notebook 5 of 6 — Logistic Regression*

We built logistic regression from nothing across NB 1–4. Now we hand the work to the real
`scikit-learn` estimator and open its panel of knobs: how much to **regularize** (`C`), which **kind**
of penalty (`l1_ratio`), and how to handle **more than two classes** (softmax vs one-vs-rest). Then we
tune it honestly.

**Prerequisites.** NB 1–4 (sigmoid → boundary → log-loss → gradient descent, all by hand); module 00:
cross-validation and tuning on the training set only (NB 10); chapter 01: standardization.

**What you'll be able to do.**
- Fit `LogisticRegression` and read a **regularization path** as `C` varies.
- Choose **L1 vs L2** with `l1_ratio` (and know `saga` is the solver for L1).
- Use **softmax** vs **one-vs-rest** for more than two classes.
- Tune `C` honestly with cross-validation and a single sealed test.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler

from ml_course import colors, datasets, viz

viz.use_course_style()
np.random.seed(0)

# The running set: Adelie vs Gentoo, two standardized features (as in NB 2 and NB 4).
df = datasets.load_penguins()
X2 = StandardScaler().fit_transform(df[["bill_length_mm", "flipper_length_mm"]].to_numpy(float))
y2 = (df["species"] == "Gentoo").to_numpy().astype(int)
print(f"2-feature binary: {X2.shape}, Gentoo {int(y2.sum())}/{len(y2)}")

## Where we are

Across NB 1–4 we built logistic regression from the ground up: a linear score, a decision boundary, the
log-loss, and the gradient descent that minimizes it. `scikit-learn` packages all of that into one
object, `LogisticRegression`. Its defaults add a touch of regularization, so we turn that off (`C=np.inf`) to confirm it is the **same** model we built; then we explore the
parameters it exposes.

In [ ]:
# Fit the library's estimator with no regularization (C = inf), our by-hand setting from NB 4.
parity = LogisticRegression(C=np.inf, max_iter=200000).fit(X2, y2)
print("LogisticRegression(C=inf) on standardized bill + flipper:")
print(f"  coef_      = {parity.coef_.ravel().round(3)}")
print(f"  intercept_ = {parity.intercept_[0]:.3f}")
print(f"  accuracy   = {parity.score(X2, y2):.4f}")
# NB 4 showed by-hand gradient descent lands on exactly these weights -> same model.

## The estimator's knobs (the current scikit-learn API)

These are the parameters we will use. The regularization API changed recently, so it is worth stating
the **current** spelling plainly:

| knob | what it does | values |
|---|---|---|
| `C` | inverse regularization **strength** | small `C` = strong penalty (small weights); `C=np.inf` = none |
| `l1_ratio` | the **kind** of penalty | `0` = L2 / ridge (**default**); `1` = L1 / lasso; between = elastic-net |
| `solver` | the optimizer | `lbfgs` (default, L2 only); **`saga`** for L1 / elastic-net |
| multi-class | for 3+ classes | **softmax** automatically; `OneVsRestClassifier(...)` for one-vs-rest |

Two recent changes worth knowing: the old `penalty=` argument is **deprecated** (it disappears in 1.10)
— set the penalty's *kind* with `l1_ratio` instead; and the old `multi_class=` argument is **gone** —
softmax is automatic for three or more classes.

## Regularization, and the `C` dial

Left unchecked, the weights can grow large — a very steep, very confident boundary that fits the
training noise (and, as we will see, can run away entirely). **Regularization** adds a penalty on the
size of the weights, keeping them modest. The strength of that penalty is set by `C`, and the
convention is **inverse**:

- **small `C`** → strong penalty → small, smooth weights;
- **large `C`** → weak penalty → weights free to grow.

Let us watch the weights respond as we sweep `C`.

In [ ]:
# Four numeric measurements, binary Adelie vs Gentoo, standardized.
full = datasets.load_penguins_full()
num4 = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
bin4 = full[full["species"].isin(["Adelie", "Gentoo"])][num4 + ["species"]].dropna()
X4 = StandardScaler().fit_transform(bin4[num4].to_numpy(float))
y4 = (bin4["species"] == "Gentoo").to_numpy().astype(int)

Cs = np.logspace(-2, 4, 25)
coefs = np.array([LogisticRegression(C=C, max_iter=200000).fit(X4, y4).coef_.ravel() for C in Cs])
norms = np.linalg.norm(coefs, axis=1)

print("L2 path, ||w||2 at a few C:")
for C in [0.01, 0.1, 1, 100]:
    w = LogisticRegression(C=C, max_iter=200000).fit(X4, y4).coef_
    print(f"  C={C:>6}: ||w||2 = {np.linalg.norm(w):.2f}")

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.4))
axL.plot(Cs, norms, color=colors.COLORS["model"], linewidth=2, marker="o", markersize=3)
axL.set_xscale("log")
axL.set_xlabel("C  (inverse regularization strength)")
axL.set_ylabel("||w||2  (total weight size)")
axL.set_title("L2 regularization path")
short = ["bill_length", "bill_depth", "flipper", "body_mass"]
for j, name in enumerate(short):
    axR.plot(
        Cs,
        coefs[:, j],
        linewidth=2,
        marker="o",
        markersize=3,
        label=name,
        color=colors.CLASS_CYCLE[j],
    )
axR.axhline(0, color=colors.COLORS["grid"], linewidth=1)
axR.set_xscale("log")
axR.set_xlabel("C")
axR.set_ylabel("coefficient")
axR.set_title("each weight vs C")
axR.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Read the figure.** *Left:* as `C` grows (the penalty weakens), the total weight size ‖w‖₂ climbs
from about 0.8 to roughly 8.5 and then appears to flatten. Small `C` pulls every weight smoothly toward 0.
*Right:* the four coefficients grow together, each keeping its sign. On *this* data the four
measurements separate Adélie from Gentoo **perfectly**, so the weights never truly settle — the apparent
flattening is the solver **converging** on an increasingly flat loss surface, not a real bottom. The
next section is *why*.

## Why regularization exists: separation → divergence

Here is the catch with a perfectly separable problem. If a line can split the classes with **no**
mistakes, the model can always lower the log-loss a little more by scaling the weights **up** — a
sharper boundary makes the right-side probabilities even closer to 1 and the wrong-side ones even closer
to 0. There is no point where it has to stop: the best weights are **infinite**. A penalty (a finite
`C`) is what gives the problem a finite answer.

We can see it with a single penguin. Our two-feature set has exactly **one** Adélie/Gentoo pair that
overlaps; with it, the fit is finite. Remove that one point and the rest is perfectly separable.

In [ ]:
# The one misclassified point makes the full set non-separable (finite fit).
keep = parity.predict(X2) == y2
Xsep, ysep = X2[keep], y2[keep]
print(f"full set: {len(y2)} points (1 overlapping) | separable slice: {int(keep.sum())} points")

grid_C = np.logspace(0, 8, 25)
norm_full, norm_sep = [], []
for C in grid_C:
    norm_full.append(np.linalg.norm(LogisticRegression(C=C, max_iter=200000).fit(X2, y2).coef_))
    norm_sep.append(np.linalg.norm(LogisticRegression(C=C, max_iter=200000).fit(Xsep, ysep).coef_))

# The plateau is convergence, not the iteration cap: the solver succeeds in a dozen steps.
sep_top = LogisticRegression(C=1e8, max_iter=200000).fit(Xsep, ysep)
print(f"separable slice at C=1e8: ||w|| = {np.linalg.norm(sep_top.coef_):.0f}, "
      f"converged in {sep_top.n_iter_[0]} iterations (of a 200000 limit)")

fig, ax = plt.subplots(figsize=(7, 4.6))
ax.plot(grid_C, norm_full, color=colors.COLORS["model"], linewidth=2, marker="o", markersize=3,
        label="full set (1 overlap) — settles")
ax.plot(grid_C, norm_sep, color=colors.COLORS["error"], linewidth=2, marker="o", markersize=3,
        label="that point removed — runs away")
ax.set_xscale("log")
ax.set_xlabel("C  (inverse regularization strength)")
ax.set_ylabel("||w||  (weight size)")
ax.set_title("Perfect separation makes the weights run away")
ax.legend()
plt.show()

**Read the figure.** One penguin decides it. With the overlapping pair kept (blue), the weights settle
to a finite fit (‖w‖ ≈ 11) once `C` is large enough — there is a real best answer. Remove that single
point (coral) and the weights **run away** as `C` grows (to ‖w‖ ≈ 29, where the solver **converges** in a dozen steps on a loss surface gone flat — not for want of iterations) — no finite best
answer exists. This is why a real pipeline keeps a little regularization: it guarantees a sane,
finite fit even when the data is (nearly) separable.

## L1 vs L2: the kind of penalty (`l1_ratio`)

There are two classic penalties, and `l1_ratio` chooses between them:

- **L2** (ridge, `l1_ratio=0`): penalize the sum of *squared* weights. It shrinks **all** weights
  smoothly toward 0, but none reach exactly 0.
- **L1** (lasso, `l1_ratio=1`): penalize the sum of *absolute* weights. It drives **some weights to
  exactly 0** — switching features off. That is automatic **feature selection**.

L1 needs the **`saga`** solver (the default `lbfgs` handles L2 only). Let us give the model four real
features and four columns of pure noise, and see what each penalty does.

In [ ]:
rng = np.random.RandomState(0)
X4n = np.hstack([X4, StandardScaler().fit_transform(rng.randn(len(y4), 4))])
feat8 = ["bill_length", "bill_depth", "flipper", "body_mass"] + [f"noise{i}" for i in range(1, 5)]

clf_l1 = LogisticRegression(l1_ratio=1.0, solver="saga", C=1.0, max_iter=50000)
w_l1 = clf_l1.fit(X4n, y4).coef_.ravel()
w_l2 = LogisticRegression(C=1.0, max_iter=10000).fit(X4n, y4).coef_.ravel()
print(f"L1 nonzero weights: {int(np.sum(np.abs(w_l1) > 1e-6))}/8")
print(f"L2 nonzero weights: {int(np.sum(np.abs(w_l2) > 1e-6))}/8")

idx = np.arange(8)
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.axvspan(3.5, 7.5, color=colors.COLORS["grid"], alpha=0.5)  # the noise region
ax.bar(idx - 0.2, np.abs(w_l2), width=0.4, color=colors.COLORS["model"], label="L2 (l1_ratio=0)")
ax.bar(
    idx + 0.2,
    np.abs(w_l1),
    width=0.4,
    color=colors.COLORS["highlight"],
    label="L1 (l1_ratio=1, saga)",
)
ax.set_xticks(idx)
ax.set_xticklabels(feat8, rotation=30, ha="right")
ax.set_ylabel("|coefficient|")
ax.set_title("L1 zeroes the noise; L2 only shrinks it")
ax.legend()
plt.tight_layout()
plt.show()

**Read the figure.** The four shaded bars on the right are the pure-noise features. **L1** (rose) sets
all four noise weights to **exactly 0** — a sparse model that has selected only the real measurements.
**L2** (blue) keeps every feature, the noise weights merely **shrunk** toward 0, never quite reaching it.
L1 selects; L2 shrinks. (On the four *real* features alone L1 keeps all four — each carries signal; only
a very strong L1, `C=0.01`, drops to a single feature.)

## More than two classes: softmax

Two classes needed one sigmoid. With three or more, `LogisticRegression` fits the **softmax**
(multinomial) model by default: one linear score per class, passed through softmax so the class
probabilities are all positive and sum to 1. Predict the class with the highest probability.

The older alternative is **one-vs-rest**: train one binary classifier per class ("this class vs all the
others"), then pick the most confident — the explicit `OneVsRestClassifier`. Let us fit both on all
three penguin species.

In [ ]:
sp3 = full[["bill_length_mm", "flipper_length_mm", "species"]].dropna()
X3 = StandardScaler().fit_transform(sp3[["bill_length_mm", "flipper_length_mm"]].to_numpy(float))
X3_df = pd.DataFrame(X3, columns=["bill_length (standardized)", "flipper_length (standardized)"])
y3 = sp3["species"].to_numpy()

cv = StratifiedKFold(5, shuffle=True, random_state=0)
cv_multi = cross_val_score(LogisticRegression(max_iter=10000), X3, y3, cv=cv).mean()
cv_ovr = cross_val_score(
    OneVsRestClassifier(LogisticRegression(max_iter=10000)), X3, y3, cv=cv
).mean()
softmax = LogisticRegression(max_iter=10000).fit(X3, y3)
ovr_pred = OneVsRestClassifier(LogisticRegression(max_iter=10000)).fit(X3, y3).predict(X3)
disagree = np.mean(softmax.predict(X3) != ovr_pred) * 100
print(f"multinomial CV acc = {cv_multi:.4f}   one-vs-rest CV acc = {cv_ovr:.4f}")
print(f"disagreement = {disagree:.1f}% | softmax coef_ shape = {softmax.coef_.shape}")

fig = viz.plot_decision_boundary(softmax, X3_df, y3)
fig.axes[0].set_title("Three species: softmax decision regions")
plt.show()

**Read the figure.** Three regions, one per species, meeting along the boundaries — each region the set
of points where that species wins the softmax. Here softmax and one-vs-rest agree on **every** prediction
(0 % disagreement) and score the same under cross-validation, because the three species are well
separated. They differ in **how the probabilities are formed** — softmax normalizes all three scores
together; one-vs-rest fits three independent sigmoids, then renormalizes them to sum to 1 (scikit-learn
does this inside `predict_proba`) — and on overlapping classes those
probabilities, and sometimes the predictions, would part ways.

## Choosing `C` honestly

`C` (and `l1_ratio`) are **hyperparameters**: we do not read them off the test set. The discipline from
module 00 (NB 10) holds — pick them by **cross-validation on the training data**, then look at the test
set exactly **once**. `GridSearchCV` automates the search.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X2, y2, test_size=0.3, random_state=0, stratify=y2)
grid = GridSearchCV(
    LogisticRegression(max_iter=10000),
    {"C": np.logspace(-2, 3, 12)},
    cv=StratifiedKFold(5, shuffle=True, random_state=0),
)
grid.fit(Xtr, ytr)
print(f"train {len(ytr)} / test {len(yte)} penguins")
print(f"best C = {grid.best_params_['C']:.3g}   (cross-validated accuracy {grid.best_score_:.4f})")
print(f"sealed test accuracy = {grid.score(Xte, yte):.4f}")

**Read the result.** Cross-validation on the training set favoured a **moderately regularized** model,
`C ≈ 0.08` (CV accuracy ≈ 0.995): stronger regularization (`C = 0.01`) under-fits at ≈ 0.97, and weaker
(large `C`) is slightly worse. Touched **once**, the sealed test set scores **1.000** (all 83 held-out
penguins correct) — this two-feature split is an easy problem, so do not over-read the perfect score; NB
6 is the demanding case. The point is the **procedure**: search on train, decide by cross-validation,
test once.

## Your turn

1. **Trace the path.** From the L2 regularization path (Figure A), which `C` gives the **smallest**
   weights? Is that the same `C` cross-validation chose for accuracy? What does that tell you about
   "smaller weights" versus "better predictions"?
2. **Zero a real feature.** Refit the L1 model (`l1_ratio=1`, `solver="saga"`) on the four real features,
   lowering `C` toward `0.01`. Which feature survives longest, and why might that one carry the most
   signal?
3. **Softmax vs one-vs-rest.** Take one borderline three-species penguin (near a boundary in Figure D)
   and print its class probabilities from `softmax.predict_proba` and from the `OneVsRestClassifier`. Do
   they rank the species the same way? Do both sets sum to 1?

## What you built

- `LogisticRegression` and its knobs — confirmed it is the model we built by hand (NB 4 parity).
- **`C`**: the regularization dial; the **regularization path**; and **separation → divergence**, the
  reason regularization exists.
- **`l1_ratio`**: **L1** (lasso) selects features (weights hit exactly 0; needs `saga`); **L2** (ridge)
  shrinks them smoothly.
- **softmax vs one-vs-rest** for more than two classes, and **honest tuning** with `GridSearchCV` and a
  single sealed test.

**Vocabulary.** *regularization* · *`C` (inverse strength)* · *L1 / lasso* · *L2 / ridge* · *`l1_ratio`*
· *solver (`saga`)* · *softmax / multinomial* · *one-vs-rest* · *hyperparameter*.

You can now drive the real estimator with intent. Next we put it to work on a problem where the stakes
are real.

## Going further (optional)

- **Elastic-net** (`l1_ratio` strictly between 0 and 1) blends L1's selection with L2's smooth
  shrinkage — useful when features are correlated.
- **`class_weight="balanced"`** re-weights the loss for an imbalanced problem, so the rare class is not
  ignored (NB 6 meets imbalance for real).
- The **solver** matters: `lbfgs` (fast, L2 only), `saga` (L1 / elastic-net, large data), `liblinear`
  (small data). The penalty you want decides the solver you can use.

A look ahead: **NB 6** is the demanding case — breast-cancer diagnosis — where calibrated probabilities
and a deliberately chosen threshold decide who gets flagged.

## References

- Cox, D. R. (1958). *The regression analysis of binary sequences.* J. R. Stat. Soc. B, 20(2), 215–242.
  DOI: 10.1111/j.2517-6161.1958.tb00292.x
- Hoerl, A. E., & Kennard, R. W. (1970). *Ridge regression: biased estimation for nonorthogonal
  problems.* Technometrics, 12(1), 55–67. DOI: 10.1080/00401706.1970.10488634
- Tibshirani, R. (1996). *Regression shrinkage and selection via the lasso.* J. R. Stat. Soc. B, 58(1),
  267–288. DOI: 10.1111/j.2517-6161.1996.tb02080.x
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (§3.4, §4.4).
  DOI: 10.1007/978-0-387-84858-7

---

*Previous: 04 — Fitting II: gradient descent*  ·  *Next: 06 — A demanding case: breast cancer*